In [1]:
# ============================================================
# Cell 1: Setup
# ============================================================
# Standard imports plus type-hints support for clean data structures.

from dataclasses import dataclass, field
from typing import Optional
import numpy as np

# Mount Drive (if you want to load Basic Pitch outputs from previous notebooks)
from google.colab import drive
drive.mount('/content/drive')

print("Setup complete.")

Mounted at /content/drive
Setup complete.


In [2]:
# ============================================================
# Cell 2: Fretboard Lookup Table
# ============================================================
# The foundational data structure. Every (string, fret) pair maps
# deterministically to a MIDI pitch, and every MIDI pitch has a
# known set of valid (string, fret) positions on the guitar.
#
# String indexing convention:
#   0 = low E (thickest string, lowest pitch) — open MIDI = 40
#   5 = high E (thinnest string, highest pitch) — open MIDI = 64
# This matches GuitarSet's convention.

# Standard tuning — open-string MIDI values
OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]
MAX_FRET = 22  # most acoustic and electric guitars

# Pitch class names for display
PITCH_CLASSES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']


def fret_to_midi(string: int, fret: int) -> int:
    """Convert a (string, fret) position to its MIDI pitch.

    Example: fret_to_midi(5, 0) -> 64 (high E open)
             fret_to_midi(0, 3) -> 43 (low E string, fret 3 = G2)
    """
    if not 0 <= string <= 5:
        raise ValueError(f"String must be 0-5, got {string}")
    if not 0 <= fret <= MAX_FRET:
        raise ValueError(f"Fret must be 0-{MAX_FRET}, got {fret}")
    return OPEN_STRING_MIDI[string] + fret


def midi_to_positions(midi: int) -> list[tuple[int, int]]:
    """Return all valid (string, fret) positions that produce this MIDI pitch.

    Example: midi_to_positions(64) -> [(2, 14), (3, 9), (4, 5), (5, 0)]
        E4 can be played on the D string fret 14, G string fret 9,
        B string fret 5, or high E string open.
    """
    positions = []
    for string in range(6):
        fret = midi - OPEN_STRING_MIDI[string]
        if 0 <= fret <= MAX_FRET:
            positions.append((string, fret))
    return positions


def midi_to_note_name(midi: int) -> str:
    """Convert MIDI number to human-readable note name (e.g. 64 -> 'E4')."""
    octave = midi // 12 - 1
    pc = PITCH_CLASSES[midi % 12]
    return f"{pc}{octave}"


# Smoke test
print("Fretboard lookup test:\n")

print("Open strings (low to high):")
for s in range(6):
    midi = fret_to_midi(s, 0)
    print(f"  String {s}: MIDI {midi} = {midi_to_note_name(midi)}")

print("\nAll positions for E4 (MIDI 64):")
for s, f in midi_to_positions(64):
    print(f"  String {s} (open {midi_to_note_name(OPEN_STRING_MIDI[s])}), fret {f}")

print("\nAll positions for A2 (MIDI 45):")
for s, f in midi_to_positions(45):
    print(f"  String {s} (open {midi_to_note_name(OPEN_STRING_MIDI[s])}), fret {f}")

Fretboard lookup test:

Open strings (low to high):
  String 0: MIDI 40 = E2
  String 1: MIDI 45 = A2
  String 2: MIDI 50 = D3
  String 3: MIDI 55 = G3
  String 4: MIDI 59 = B3
  String 5: MIDI 64 = E4

All positions for E4 (MIDI 64):
  String 1 (open A2), fret 19
  String 2 (open D3), fret 14
  String 3 (open G3), fret 9
  String 4 (open B3), fret 5
  String 5 (open E4), fret 0

All positions for A2 (MIDI 45):
  String 0 (open E2), fret 5
  String 1 (open A2), fret 0


In [3]:
# ============================================================
# Cell 3: TabNote Data Structure
# ============================================================
# The unified data structure for notes that have been assigned
# to specific fretboard positions. This matches the shape of
# GuitarSet's ground-truth annotations, so evaluation is direct.

@dataclass
class TabNote:
    """A note with its assigned position on the fretboard."""
    start: float           # seconds
    duration: float        # seconds
    midi: int              # MIDI pitch (60 = middle C, 64 = E4, etc.)
    string: int            # 0 (low E) to 5 (high E)
    fret: int              # 0 (open) to MAX_FRET
    confidence: float = 1.0  # how sure the algorithm is about this assignment

    @property
    def note_name(self) -> str:
        return midi_to_note_name(self.midi)

    def __repr__(self) -> str:
        return (f"TabNote(t={self.start:.2f}s, {self.note_name}, "
                f"string={self.string}, fret={self.fret})")


# Smoke test
note = TabNote(start=1.5, duration=0.4, midi=64, string=3, fret=9)
print(f"Created: {note}")
print(f"  Pitch verification: fret_to_midi({note.string}, {note.fret}) = "
      f"{fret_to_midi(note.string, note.fret)}, matches midi={note.midi}: "
      f"{fret_to_midi(note.string, note.fret) == note.midi}")

Created: TabNote(t=1.50s, E4, string=3, fret=9)
  Pitch verification: fret_to_midi(3, 9) = 64, matches midi=64: True


In [4]:
# ============================================================
# Cell 4: Naive Baseline Fret Assignment
# ============================================================
# For each note, pick the position with the lowest fret number.
# This is intentionally simple

def naive_assign(notes: list[dict]) -> list[TabNote]:
    """Assign fret positions using the simplest possible rule:
    pick the lowest-fret valid position for each note independently.

    Args:
        notes: list of dicts with keys 'start', 'duration', 'midi'
               (the format from Basic Pitch's note detection)

    Returns:
        list of TabNote objects with assigned positions
    """
    tab_notes = []
    for n in notes:
        positions = midi_to_positions(n['midi'])
        if not positions:
            # Note is outside guitar range — skip
            continue

        # Pick lowest-fret position. Ties broken by lowest string.
        best = min(positions, key=lambda p: (p[1], p[0]))

        tab_notes.append(TabNote(
            start=n['start'],
            duration=n['duration'],
            midi=n['midi'],
            string=best[0],
            fret=best[1],
            confidence=0.5,  # low confidence — we're just guessing
        ))
    return tab_notes


# Smoke test — make up a simple riff and see how naive assignment handles it
test_notes = [
    {'start': 0.0, 'duration': 0.5, 'midi': 64},  # E4
    {'start': 0.5, 'duration': 0.5, 'midi': 67},  # G4
    {'start': 1.0, 'duration': 0.5, 'midi': 71},  # B4
    {'start': 1.5, 'duration': 0.5, 'midi': 76},  # E5
]

naive_tab = naive_assign(test_notes)
print("Naive assignment of E4 → G4 → B4 → E5:")
for tn in naive_tab:
    print(f"  {tn}")

Naive assignment of E4 → G4 → B4 → E5:
  TabNote(t=0.00s, E4, string=5, fret=0)
  TabNote(t=0.50s, G4, string=5, fret=3)
  TabNote(t=1.00s, B4, string=5, fret=7)
  TabNote(t=1.50s, E5, string=5, fret=12)


In [5]:
# ============================================================
# Cell 5: Music Theory Helpers
# ============================================================
# Functions that encode music theory knowledge — what notes are
# in a key, what notes are chord tones, etc.

# Scale degree patterns (semitone intervals from tonic)
MAJOR_SCALE_INTERVALS = [0, 2, 4, 5, 7, 9, 11]
MINOR_SCALE_INTERVALS = [0, 2, 3, 5, 7, 8, 10]


def note_name_to_pitch_class(note_name: str) -> int:
    """Convert note name to pitch class number (0-11).
    Handles both sharps and flats: 'C#' = 'Db' = 1
    """
    # Normalize flats to sharps
    flat_to_sharp = {'Db': 'C#', 'Eb': 'D#', 'Gb': 'F#', 'Ab': 'G#', 'Bb': 'A#'}
    note_name = flat_to_sharp.get(note_name, note_name)
    return PITCH_CLASSES.index(note_name)


def notes_in_key(tonic: str, mode: str = 'major') -> set[int]:
    """Return the set of pitch classes (0-11) in a given key.

    Example: notes_in_key('D', 'major') -> {2, 4, 6, 7, 9, 11, 1}
             (D, E, F#, G, A, B, C#)
    """
    tonic_pc = note_name_to_pitch_class(tonic)
    intervals = MAJOR_SCALE_INTERVALS if mode == 'major' else MINOR_SCALE_INTERVALS
    return {(tonic_pc + i) % 12 for i in intervals}


def is_in_key(midi: int, tonic: str, mode: str = 'major') -> bool:
    """Check if a MIDI note belongs to the given key's scale."""
    return (midi % 12) in notes_in_key(tonic, mode)


# Chord-tone definitions for the most common chord qualities
# Expressed as semitone intervals from the chord root
CHORD_INTERVALS = {
    'maj':  [0, 4, 7],          # major triad
    'min':  [0, 3, 7],          # minor triad
    '7':    [0, 4, 7, 10],      # dominant 7th
    'maj7': [0, 4, 7, 11],      # major 7th
    'min7': [0, 3, 7, 10],      # minor 7th
    'dim':  [0, 3, 6],          # diminished triad
}


def parse_chord_label(label: str) -> Optional[tuple[str, str]]:
    """Parse a chord label like 'D#:maj' or 'F#:min' into (root, quality).
    Returns None for non-chord labels like 'N'.
    """
    if not label or label == 'N':
        return None
    if ':' in label:
        root, quality = label.split(':', 1)
    else:
        # No quality specified — assume major
        root, quality = label, 'maj'
    # Normalize 'major'/'minor' to 'maj'/'min'
    quality = quality.replace('major', 'maj').replace('minor', 'min')
    return root, quality


def chord_tones(label: str) -> set[int]:
    """Return the set of pitch classes that are chord tones for this chord.

    Example: chord_tones('C:maj') -> {0, 4, 7}  (C, E, G)
             chord_tones('D:min') -> {2, 5, 9}  (D, F, A)
    """
    parsed = parse_chord_label(label)
    if parsed is None:
        return set()
    root, quality = parsed
    root_pc = note_name_to_pitch_class(root)
    intervals = CHORD_INTERVALS.get(quality, CHORD_INTERVALS['maj'])
    return {(root_pc + i) % 12 for i in intervals}


def is_chord_tone(midi: int, chord_label: str) -> bool:
    """Check if a MIDI note is a chord tone of the given chord."""
    return (midi % 12) in chord_tones(chord_label)


# Smoke test
print("Notes in D major:")
notes = sorted(notes_in_key('D', 'major'))
print(f"  Pitch classes: {notes}")
print(f"  Note names: {[PITCH_CLASSES[pc] for pc in notes]}")

print("\nChord tones of F# minor:")
tones = sorted(chord_tones('F#:min'))
print(f"  Pitch classes: {tones}")
print(f"  Note names: {[PITCH_CLASSES[pc] for pc in tones]}")

print("\nIs E4 (MIDI 64) in D major?", is_in_key(64, 'D', 'major'))
print("Is E4 a chord tone of Em?", is_chord_tone(64, 'E:min'))
print("Is F4 (MIDI 65) in D major?", is_in_key(65, 'D', 'major'))

Notes in D major:
  Pitch classes: [1, 2, 4, 6, 7, 9, 11]
  Note names: ['C#', 'D', 'E', 'F#', 'G', 'A', 'B']

Chord tones of F# minor:
  Pitch classes: [1, 6, 9]
  Note names: ['C#', 'F#', 'A']

Is E4 (MIDI 64) in D major? True
Is E4 a chord tone of Em? True
Is F4 (MIDI 65) in D major? False


In [6]:
# ============================================================
# Cell 6: Chord Context Lookup
# ============================================================

def chord_at_time(time: float, chord_progression: list[tuple]) -> Optional[str]:
    """Find which chord (if any) is active at a given time.

    Args:
        time: time in seconds
        chord_progression: list of (start, end, label) tuples

    Returns:
        chord label string, or None if no chord is active at that time
    """
    for start, end, label in chord_progression:
        if start <= time < end:
            return label
    return None


# Smoke test
fake_progression = [
    (0.0,  4.0, 'D:maj'),
    (4.0,  8.0, 'F#:min'),
    (8.0, 12.0, 'E:maj'),
]

print("Chord at various times:")
for t in [0.5, 2.0, 4.0, 6.5, 10.0, 15.0]:
    chord = chord_at_time(t, fake_progression)
    print(f"  t={t}s: {chord}")

Chord at various times:
  t=0.5s: D:maj
  t=2.0s: D:maj
  t=4.0s: F#:min
  t=6.5s: F#:min
  t=10.0s: E:maj
  t=15.0s: None


In [9]:
# ============================================================
# Cell 7: Music-Theory-Aware Fret Assignment (v1)
# ============================================================
# For each note, score every valid (string, fret) position
# based on music theory and playing context, then pick the best.

def score_position(
    midi: int,
    candidate: tuple[int, int],
    detected_key: tuple[str, str],
    current_chord: Optional[str],
    previous_position: Optional[tuple[int, int]],
    weights: Optional[dict] = None,
) -> float:
    """Score a candidate (string, fret) position. Higher = better."""
    if weights is None:
        weights = {
            'key_alignment':       1.0,
            'chord_tone':          2.0,
            'open_string_bonus':   1.0,
            'low_position_bonus':  0.5,    # frets 0-3 are "home"
            'middle_neck_bonus':   0.3,    # frets 4-12 are also comfortable
            'position_continuity': 0.5,    # softer than before
            'continuity_cap':      5.0,    # max fret distance that's penalized
        }

    string, fret = candidate
    score = 0.0

    tonic, mode = detected_key
    if is_in_key(midi, tonic, mode):
        score += weights['key_alignment']

    if current_chord is not None and is_chord_tone(midi, current_chord):
        score += weights['chord_tone']

    # Position comfort bonuses
    if fret == 0:
        score += weights['open_string_bonus']
    elif fret <= 3:
        score += weights['low_position_bonus']
    elif 4 <= fret <= 12:
        score += weights['middle_neck_bonus']

    # Position continuity — softer and capped, so big jumps are penalized
    # but not catastrophically. Square root grows slower than linear.
    if previous_position is not None:
        prev_fret = previous_position[1]
        if prev_fret > 0 and fret > 0:
            fret_distance = min(abs(fret - prev_fret), weights['continuity_cap'])
            score -= weights['position_continuity'] * (fret_distance ** 0.5)

    return score


def music_theory_assign(
    notes: list[dict],
    detected_key: tuple[str, str],
    chord_progression: list[tuple],
) -> list[TabNote]:
    """Music-theory-aware fret assignment.

    Args:
        notes: list of dicts with 'start', 'duration', 'midi' (from Basic Pitch)
        detected_key: (tonic, mode) tuple from key detection
        chord_progression: list of (start, end, label) from chord detection

    Returns:
        list of TabNote with assigned positions
    """
    tab_notes = []
    previous_position = None

    for n in notes:
        positions = midi_to_positions(n['midi'])
        if not positions:
            continue

        current_chord = chord_at_time(n['start'], chord_progression)

        # Score every candidate position
        scored = [
            (pos, score_position(n['midi'], pos, detected_key,
                                  current_chord, previous_position))
            for pos in positions
        ]
        # Pick highest-scoring
        best_pos, best_score = max(scored, key=lambda x: x[1])

        tab_notes.append(TabNote(
            start=n['start'],
            duration=n['duration'],
            midi=n['midi'],
            string=best_pos[0],
            fret=best_pos[1],
            confidence=best_score / 5.0,  # rough normalization for display
        ))
        previous_position = best_pos

    return tab_notes


# Smoke test — same notes as the naive version, but now with musical context
test_notes = [
    {'start': 0.0, 'duration': 0.5, 'midi': 64},  # E4
    {'start': 0.5, 'duration': 0.5, 'midi': 67},  # G4
    {'start': 1.0, 'duration': 0.5, 'midi': 71},  # B4
    {'start': 1.5, 'duration': 0.5, 'midi': 76},  # E5
]
test_key = ('E', 'minor')
test_chords = [(0.0, 2.0, 'E:min')]  # All notes happen during an Em chord

smart_tab = music_theory_assign(test_notes, test_key, test_chords)
print("Music-theory-aware assignment of E4 → G4 → B4 → E5 in E minor / Em chord:")
for tn in smart_tab:
    print(f"  {tn}  (confidence={tn.confidence:.2f})")

Music-theory-aware assignment of E4 → G4 → B4 → E5 in E minor / Em chord:
  TabNote(t=0.00s, E4, string=5, fret=0)  (confidence=0.80)
  TabNote(t=0.50s, G4, string=5, fret=3)  (confidence=0.70)
  TabNote(t=1.00s, B4, string=5, fret=7)  (confidence=0.46)
  TabNote(t=1.50s, E5, string=5, fret=12)  (confidence=0.44)


In [10]:
# ============================================================
# Cell 8: Side-by-Side Comparison
# ============================================================
# Run both algorithms on the same input and visualize the difference.

test_notes = [
    {'start': 0.0, 'duration': 0.5, 'midi': 64},  # E4
    {'start': 0.5, 'duration': 0.5, 'midi': 67},  # G4
    {'start': 1.0, 'duration': 0.5, 'midi': 71},  # B4
    {'start': 1.5, 'duration': 0.5, 'midi': 76},  # E5
    {'start': 2.0, 'duration': 0.5, 'midi': 79},  # G5
    {'start': 2.5, 'duration': 0.5, 'midi': 67},  # G4 again
]
test_key = ('E', 'minor')
test_chords = [(0.0, 3.0, 'E:min')]

naive = naive_assign(test_notes)
smart = music_theory_assign(test_notes, test_key, test_chords)

print(f"{'Note':>5}  {'Naive':>16}  {'Music-aware':>16}")
print("-" * 45)
for n_naive, n_smart in zip(naive, smart):
    naive_pos = f"str{n_naive.string} fr{n_naive.fret}"
    smart_pos = f"str{n_smart.string} fr{n_smart.fret}"
    same = "" if (n_naive.string == n_smart.string and n_naive.fret == n_smart.fret) else "  ←diff"
    print(f"{n_naive.note_name:>5}  {naive_pos:>16}  {smart_pos:>16}{same}")

 Note             Naive       Music-aware
---------------------------------------------
   E4          str5 fr0          str5 fr0
   G4          str5 fr3          str5 fr3
   B4          str5 fr7          str5 fr7
   E5         str5 fr12         str5 fr12
   G5         str5 fr15         str5 fr15
   G4          str5 fr3         str3 fr12  ←diff


In [11]:
# A test case designed to differentiate naive from smart:
# A melodic line that's "stuck" in a high-position D major run.
# Naive will scatter notes across multiple strings at low frets.
# Smart should keep them clustered on adjacent strings in one position.

# All notes are in D major and could be chord tones of D, F#m, or G
# But they're played fast, so a real guitarist would stay in one position
diagnostic_notes = [
    {'start': 0.0, 'duration': 0.2, 'midi': 74},   # D5
    {'start': 0.2, 'duration': 0.2, 'midi': 78},   # F#5
    {'start': 0.4, 'duration': 0.2, 'midi': 81},   # A5
    {'start': 0.6, 'duration': 0.2, 'midi': 78},   # F#5
    {'start': 0.8, 'duration': 0.2, 'midi': 74},   # D5
    {'start': 1.0, 'duration': 0.2, 'midi': 71},   # B4
]
test_key = ('D', 'major')
test_chords = [(0.0, 2.0, 'D:maj')]

naive_d = naive_assign(diagnostic_notes)
smart_d = music_theory_assign(diagnostic_notes, test_key, test_chords)

print(f"{'Note':>5}  {'Naive':>16}  {'Music-aware':>16}")
print("-" * 45)
for n_naive, n_smart in zip(naive_d, smart_d):
    naive_pos = f"str{n_naive.string} fr{n_naive.fret}"
    smart_pos = f"str{n_smart.string} fr{n_smart.fret}"
    same = "" if (n_naive.string == n_smart.string and n_naive.fret == n_smart.fret) else "  ←diff"
    print(f"{n_naive.note_name:>5}  {naive_pos:>16}  {smart_pos:>16}{same}")

 Note             Naive       Music-aware
---------------------------------------------
   D5         str5 fr10         str5 fr10
  F#5         str5 fr14         str5 fr14
   A5         str5 fr17         str5 fr17
  F#5         str5 fr14         str4 fr19  ←diff
   D5         str5 fr10         str3 fr19  ←diff
   B4          str5 fr7         str2 fr21  ←diff
